In [32]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import mutual_info_score
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression

In [33]:
#Data preparation
df = pd.read_csv('https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/course_lead_scoring_2026.csv')

df.columns = df.columns.str.lower().str.replace(' ', '_')
categorical_columns = list(df.select_dtypes(include=['object', 'str']).columns) #added 'str' because otherwise it returned []
numerical_columns = list(df.select_dtypes(include='number').columns) #had to look select_dtypes up
for c in categorical_columns:
    df[c] = df[c].str.strip().str.lower().str.replace(' ', '_')
    df[c] = df[c].replace('', np.nan)

df[categorical_columns] = df[categorical_columns].fillna('NA')
df[numerical_columns] = df[numerical_columns].fillna(0.0)

In [34]:
#Q1
df.industry.value_counts() #could also be df.industry.mode()

industry
technology       1173
retail            925
healthcare        840
finance           720
education         639
manufacturing     463
NA                240
Name: count, dtype: int64

In [35]:
#Q2
df[numerical_columns].corr() #looked .corr() up
#to check up: df[numerical_columns].corr().unstack().abs().sort_values(ascending=False)

,annual_income,number_of_courses_viewed,interaction_count,lead_score,converted
annual_income,1.000000,0.161300,0.122842,0.229496,0.189652
number_of_courses_viewed,0.161300,1.000000,0.721609,0.757204,0.395038
interaction_count,0.122842,0.721609,1.000000,0.915746,0.448624
lead_score,0.229496,0.757204,0.915746,1.000000,0.483246
converted,0.189652,0.395038,0.448624,0.483246,1.000000


In [36]:
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=42)
df_train, df_val = train_test_split(df_full_train, test_size=0.25, random_state=42)

df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)

y_train = df_train.converted.values
y_val = df_val.converted.values
y_test = df_test.converted.values

del df_train['converted']
del df_val['converted']
del df_test['converted']

In [37]:
#Q3
def mutual_info_converted_score(series):
    return mutual_info_score(series, df_full_train.converted)
mi = df_full_train[categorical_columns].apply(mutual_info_converted_score)
round(mi.sort_values(ascending=False), 2)

lead_source          0.03
employment_status    0.02
industry             0.00
location             0.00
dtype: float64

In [38]:
#Q4
numerical_columns = ['annual_income','number_of_courses_viewed','interaction_count','lead_score'] #so converted is not in the list

dv = DictVectorizer(sparse=False)
train_dict = df_train[categorical_columns + numerical_columns].to_dict(orient='records')
X_train = dv.fit_transform(train_dict)
val_dict = df_val[categorical_columns + numerical_columns].to_dict(orient='records')
X_val = dv.transform(val_dict)

model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
model.fit(X_train, y_train)

y_pred = model.predict(X_val)
accuracy = (y_pred == y_val).mean()
round(accuracy, 2)

np.float64(0.64)

In [39]:
#Q5
original_accuracy = (model.predict(X_val) == y_val).mean()

features = categorical_columns + numerical_columns
features_to_eliminate = ['lead_source', 'number_of_courses_viewed', 'interaction_count']

for feature in features_to_eliminate:
    reduced_features = [f for f in features if f != feature]

    train_dict = df_train[reduced_features].to_dict(orient='records')
    val_dict = df_val[reduced_features].to_dict(orient='records')

    dv = DictVectorizer(sparse=False)
    X_train = dv.fit_transform(train_dict)
    X_val = dv.transform(val_dict)

    model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)

    model.fit(X_train, y_train)
    accuracy = (model.predict(X_val) == y_val).mean()

    print(feature, original_accuracy - accuracy)

lead_source 0.0030000000000000027
number_of_courses_viewed 0.0020000000000000018
interaction_count 0.04400000000000004


In [42]:
#Q6
parameters_c = [0.000001, 0.00001, 0.0001, 0.001]
for C_parameter in parameters_c:
    train_dict = df_train[features].to_dict(orient='records')
    val_dict = df_val[features].to_dict(orient='records')

    dv = DictVectorizer(sparse=False)
    X_train = dv.fit_transform(train_dict)
    X_val = dv.transform(val_dict)

    model = LogisticRegression(solver='liblinear', C=C_parameter, max_iter=1000, random_state=42)

    model.fit(X_train, y_train)
    accuracy = (model.predict(X_val) == y_val).mean()
    print(C_parameter,":", round(accuracy, 3))